<a href="https://colab.research.google.com/github/hridibazaman03/220142_CNN_RPS/blob/main/220142_CNN_RPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import os
import random
import shutil
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report

print("PyTorch version:", torch.__version__)

In [ ]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================

# IMPORTANT:
# Replace this with YOUR public GitHub repository URL.

GITHUB_REPO_URL = "https://github.com/hridibazaman03/220142_CNN_RPS.git"

# Repository folder name
REPO_DIR = "/content/220142_CNN_RPS"

# Custom images inside GitHub repository
CUSTOM_DATA_DIR = os.path.join(REPO_DIR, "dataset")

# Model directory inside GitHub repository
MODEL_DIR = os.path.join(REPO_DIR, "model")

# Model filename
MODEL_FILENAME = "rps_cnn.pth"

# Full model path
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_FILENAME)

# Dataset location
DATA_ROOT = "/content/rps_dataset"

# Image size for CNN
IMAGE_SIZE = 128

# Training settings
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 0.001

# Number of classes
NUM_CLASSES = 3

# Class names
CLASS_NAMES = ["rock", "paper", "scissors"]

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", NUM_EPOCHS)

In [ ]:
# ============================================================
# CELL 3: CLONE GITHUB REPOSITORY
# ============================================================

import subprocess

if os.path.exists(REPO_DIR):
    print("Repository already exists.")
else:
    print("Cloning repository...")

    result = subprocess.run(
        ["git", "clone", GITHUB_REPO_URL, REPO_DIR],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            "GitHub cloning failed. Check GITHUB_REPO_URL."
        )

    print("Repository cloned successfully.")

print("\nRepository contents:")
for item in os.listdir(REPO_DIR):
    print(" -", item)

In [ ]:
# ============================================================
# CELL 4: CHECK CUSTOM IMAGES
# ============================================================

if not os.path.exists(CUSTOM_DATA_DIR):
    raise FileNotFoundError(
        f"Custom dataset folder not found: {CUSTOM_DATA_DIR}"
    )

custom_images = []

valid_extensions = (".jpg", ".jpeg", ".png", ".webp")

for filename in sorted(os.listdir(CUSTOM_DATA_DIR)):
    if filename.lower().endswith(valid_extensions):
        custom_images.append(
            os.path.join(CUSTOM_DATA_DIR, filename)
        )

print("Custom images found:", len(custom_images))

for image_path in custom_images:
    print(" -", os.path.basename(image_path))

if len(custom_images) == 0:
    raise ValueError("No custom images found.")

if len(custom_images) != 10:
    print(
        "\nWARNING: The assignment asks for 10 custom images."
        f" Currently found {len(custom_images)}."
    )

In [ ]:
# ============================================================
# CELL 5: DOWNLOAD STANDARD RPS DATASET
# ============================================================

TRAIN_URL = (
    "https://storage.googleapis.com/"
    "tensorflow-1-public/course2/week4/rps.zip"
)

TEST_URL = (
    "https://storage.googleapis.com/"
    "tensorflow-1-public/course2/week4/rps-test-set.zip"
)

TRAIN_ZIP = "/content/rps_train.zip"
TEST_ZIP = "/content/rps_test.zip"

TRAIN_EXTRACT_DIR = "/content/rps_train"
TEST_EXTRACT_DIR = "/content/rps_test"

def download_file(url, output_path):
    if os.path.exists(output_path):
        print("Already downloaded:", output_path)
        return

    print("Downloading:", url)
    urllib.request.urlretrieve(url, output_path)
    print("Download complete:", output_path)


download_file(TRAIN_URL, TRAIN_ZIP)
download_file(TEST_URL, TEST_ZIP)

print("Dataset archives downloaded.")

In [ ]:
# ============================================================
# CELL 6: EXTRACT DATASET
# ============================================================

if not os.path.exists(TRAIN_EXTRACT_DIR):
    os.makedirs(TRAIN_EXTRACT_DIR)

if not os.path.exists(TEST_EXTRACT_DIR):
    os.makedirs(TEST_EXTRACT_DIR)

# Extract training dataset
with zipfile.ZipFile(TRAIN_ZIP, "r") as zip_ref:
    zip_ref.extractall(TRAIN_EXTRACT_DIR)

# Extract test dataset
with zipfile.ZipFile(TEST_ZIP, "r") as zip_ref:
    zip_ref.extractall(TEST_EXTRACT_DIR)

print("Training dataset extracted.")
print("Test dataset extracted.")

In [ ]:
# ============================================================
# CELL 7: FIND DATASET DIRECTORIES
# ============================================================

def find_class_directory(root_dir):
    """
    Find the directory containing:
        rock/
        paper/
        scissors/
    """

    for root, dirs, files in os.walk(root_dir):
        directory_names = set(d.lower() for d in dirs)

        if {"rock", "paper", "scissors"}.issubset(directory_names):
            return root

    return None


TRAIN_DATA_DIR = find_class_directory(TRAIN_EXTRACT_DIR)
TEST_DATA_DIR = find_class_directory(TEST_EXTRACT_DIR)

if TRAIN_DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find rock/paper/scissors training folders."
    )

if TEST_DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find rock/paper/scissors test folders."
    )

print("Training directory:")
print(TRAIN_DATA_DIR)

print("\nTest directory:")
print(TEST_DATA_DIR)

In [ ]:
# ============================================================
# CELL 8: DATA PREPROCESSING
# ============================================================

# ImageNet-style normalization is commonly used for RGB CNNs.
# Most importantly, EXACTLY THE SAME values must be used
# for training and custom prediction.

NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    # Data augmentation ONLY for training
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),

    transforms.ToTensor(),
    transforms.Normalize(
        mean=NORMALIZE_MEAN,
        std=NORMALIZE_STD
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=NORMALIZE_MEAN,
        std=NORMALIZE_STD
    )
])

print("Transforms created successfully.")

In [ ]:
# ============================================================
# CELL 9: LOAD DATASETS
# ============================================================

# Training dataset with augmentation
full_train_dataset = datasets.ImageFolder(
    root=TRAIN_DATA_DIR,
    transform=train_transform
)

# A second copy without augmentation is useful for validation
full_train_eval_dataset = datasets.ImageFolder(
    root=TRAIN_DATA_DIR,
    transform=eval_transform
)

# Official/standard test dataset
test_dataset = datasets.ImageFolder(
    root=TEST_DATA_DIR,
    transform=eval_transform
)

print("Classes:", full_train_dataset.classes)
print("Class to index:", full_train_dataset.class_to_idx)

print("\nTraining images:", len(full_train_dataset))
print("Test images:", len(test_dataset))

In [ ]:
# ============================================================
# CELL 10: TRAIN / VALIDATION SPLIT
# ============================================================

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

train_size = int(0.80 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

# Generate the same indices for both transformed datasets
indices = torch.randperm(
    len(full_train_dataset),
    generator=torch.Generator().manual_seed(SEED)
).tolist()

train_indices = indices[:train_size]
val_indices = indices[train_size:]

# Subsets
train_dataset = torch.utils.data.Subset(
    full_train_dataset,
    train_indices
)

val_dataset = torch.utils.data.Subset(
    full_train_eval_dataset,
    val_indices
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
# ============================================================
# CELL 11: DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

print("DataLoaders created.")

In [ ]:
# ============================================================
# CELL 12: VISUALIZE TRAINING DATA
# ============================================================

def denormalize(image):
    mean = torch.tensor(NORMALIZE_MEAN).view(3, 1, 1)
    std = torch.tensor(NORMALIZE_STD).view(3, 1, 1)

    image = image.cpu() * std + mean

    return torch.clamp(image, 0, 1)


images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 8))

for i in range(min(12, len(images))):
    plt.subplot(3, 4, i + 1)

    image = denormalize(images[i])

    plt.imshow(image.permute(1, 2, 0))
    plt.title(CLASS_NAMES[labels[i].item()])
    plt.axis("off")

plt.suptitle("Sample RPS Training Images")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 13: CNN MODEL
# ============================================================

class CNN(nn.Module):

    def __init__(self, num_classes=3):
        super(CNN, self).__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Block 2
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Block 3
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        # Makes the model independent of exact spatial size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, num_classes)
        )


    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = self.classifier(x)

        return x


model = CNN(NUM_CLASSES).to(DEVICE)

print(model)

In [ ]:
# ============================================================
# CELL 14: LOSS + OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

In [ ]:
# ============================================================
# CELL 15: TRAINING FUNCTIONS
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy


def evaluate(model, loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
# ============================================================
# CELL 16: TRAIN OR LOAD MODEL
# ============================================================

os.makedirs(MODEL_DIR, exist_ok=True)

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

best_val_accuracy = 0.0

# ------------------------------------------------------------
# OPTION 1: LOAD SAVED MODEL
# ------------------------------------------------------------

if os.path.exists(MODEL_PATH):

    print("=" * 60)
    print("SAVED MODEL FOUND")
    print("=" * 60)

    model.load_state_dict(
        torch.load(
            MODEL_PATH,
            map_location=DEVICE
        )
    )

    model.to(DEVICE)

    print("Model loaded from:")
    print(MODEL_PATH)


# ------------------------------------------------------------
# OPTION 2: TRAIN MODEL
# ------------------------------------------------------------

else:

    print("=" * 60)
    print("NO SAVED MODEL FOUND")
    print("TRAINING CNN...")
    print("=" * 60)

    for epoch in range(NUM_EPOCHS):

        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer
        )

        val_loss, val_accuracy = evaluate(
            model,
            val_loader,
            criterion
        )

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)

        print(
            f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.2f}%"
        )

        # Save best model
        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy

            torch.save(
                model.state_dict(),
                MODEL_PATH
            )

            print(
                f"  -> Best model saved "
                f"({best_val_accuracy:.2f}%)"
            )

    # Load best model after training
    model.load_state_dict(
        torch.load(
            MODEL_PATH,
            map_location=DEVICE
        )
    )

    print("\nTraining complete.")
    print("Best validation accuracy:",
          f"{best_val_accuracy:.2f}%")
    print("Model saved to:", MODEL_PATH)

In [ ]:
# ============================================================
# CELL 17: TRAINING HISTORY
# ============================================================

if len(train_losses) == 0:
    print(
        "Training history is not available because "
        "the model was loaded from an existing .pth file."
    )
else:

    epochs_range = range(1, len(train_losses) + 1)

    # -------------------------------
    # Loss graph
    # -------------------------------

    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs_range,
        train_losses,
        marker="o",
        label="Training Loss"
    )

    plt.plot(
        epochs_range,
        val_losses,
        marker="o",
        label="Validation Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


    # -------------------------------
    # Accuracy graph
    # -------------------------------

    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs_range,
        train_accuracies,
        marker="o",
        label="Training Accuracy"
    )

    plt.plot(
        epochs_range,
        val_accuracies,
        marker="o",
        label="Validation Accuracy"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("Training and Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# ============================================================
# CELL 18: STANDARD TEST SET EVALUATION
# ============================================================

test_loss, test_accuracy = evaluate(
    model,
    test_loader,
    criterion
)

print("=" * 50)
print("STANDARD TEST SET RESULTS")
print("=" * 50)

print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
# ============================================================
# CELL 19: TEST PREDICTIONS
# ============================================================

model.eval()

all_test_labels = []
all_test_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(DEVICE)

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_test_labels.extend(
            labels.numpy()
        )

        all_test_predictions.extend(
            predictions.cpu().numpy()
        )


all_test_labels = np.array(all_test_labels)
all_test_predictions = np.array(all_test_predictions)

print("Predictions collected.")
print("Number of test images:",
      len(all_test_predictions))

In [ ]:
# ============================================================
# CELL 20: CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_test_labels,
    all_test_predictions
)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("Confusion Matrix - Standard RPS Test Set")

plt.show()

In [ ]:
# ============================================================
# CELL 21: CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        all_test_labels,
        all_test_predictions,
        target_names=CLASS_NAMES,
        digits=4
    )
)

In [ ]:
# ============================================================
# CELL 22: FIND INCORRECT PREDICTIONS
# ============================================================

incorrect_indices = np.where(
    all_test_labels != all_test_predictions
)[0]

print(
    "Number of incorrect test predictions:",
    len(incorrect_indices)
)

if len(incorrect_indices) == 0:
    print(
        "Excellent! The model classified every test image correctly."
    )

In [ ]:
# ============================================================
# CELL 23: VISUAL ERROR ANALYSIS
# ============================================================

if len(incorrect_indices) > 0:

    number_to_show = min(3, len(incorrect_indices))

    selected_indices = random.sample(
        list(incorrect_indices),
        number_to_show
    )

    plt.figure(figsize=(12, 4))

    for plot_index, dataset_index in enumerate(
        selected_indices
    ):

        image, true_label = test_dataset[dataset_index]

        predicted_label = all_test_predictions[
            dataset_index
        ]

        image = denormalize(image)

        plt.subplot(
            1,
            number_to_show,
            plot_index + 1
        )

        plt.imshow(
            image.permute(1, 2, 0)
        )

        plt.title(
            f"True: {CLASS_NAMES[true_label]}\n"
            f"Pred: {CLASS_NAMES[predicted_label]}"
        )

        plt.axis("off")

    plt.suptitle(
        "Visual Error Analysis - Incorrect Predictions"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 24: CUSTOM IMAGE PREDICTION FUNCTION
# ============================================================

def predict_custom_image(image_path):

    # Load image
    image = Image.open(image_path).convert("RGB")

    # Apply exact evaluation preprocessing
    transformed_image = eval_transform(image)

    # Add batch dimension
    input_tensor = transformed_image.unsqueeze(0).to(DEVICE)

    # Evaluation mode
    model.eval()

    with torch.no_grad():

        outputs = model(input_tensor)

        # Convert logits to probabilities
        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        confidence, predicted_index = torch.max(
            probabilities,
            dim=1
        )

    predicted_class = CLASS_NAMES[
        predicted_index.item()
    ]

    confidence_percentage = (
        confidence.item() * 100
    )

    return (
        image,
        predicted_class,
        confidence_percentage,
        probabilities.cpu().numpy()[0]
    )

In [ ]:
# ============================================================
# CELL 25: PREDICT CUSTOM PHONE IMAGES
# ============================================================

custom_results = []

for image_path in custom_images:

    (
        image,
        predicted_class,
        confidence,
        probabilities
    ) = predict_custom_image(image_path)

    custom_results.append({
        "path": image_path,
        "filename": os.path.basename(image_path),
        "image": image,
        "predicted_class": predicted_class,
        "confidence": confidence,
        "probabilities": probabilities
    })


print("=" * 60)
print("CUSTOM IMAGE PREDICTIONS")
print("=" * 60)

for result in custom_results:

    print(
        f"{result['filename']:25s} -> "
        f"{result['predicted_class']:10s} "
        f"({result['confidence']:.2f}%)"
    )

In [ ]:
# ============================================================
# CELL 26: CUSTOM PREDICTION GALLERY
# ============================================================

number_of_images = len(custom_results)

columns = 5
rows = int(np.ceil(number_of_images / columns))

plt.figure(
    figsize=(18, 4 * rows)
)

for i, result in enumerate(custom_results):

    plt.subplot(
        rows,
        columns,
        i + 1
    )

    plt.imshow(result["image"])

    plt.title(
        f"Pred: {result['predicted_class']}\n"
        f"Confidence: {result['confidence']:.2f}%"
    )

    plt.axis("off")


plt.suptitle(
    "Custom Phone Image Predictions",
    fontsize=18
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 27: CUSTOM IMAGE PROBABILITIES
# ============================================================

for result in custom_results:

    print("\n" + "=" * 50)
    print(result["filename"])
    print("=" * 50)

    for class_name, probability in zip(
        CLASS_NAMES,
        result["probabilities"]
    ):

        print(
            f"{class_name:10s}: "
            f"{probability * 100:.2f}%"
        )

In [ ]:
# ============================================================
# CELL 28: FINAL SUMMARY
# ============================================================

print("=" * 65)
print("CNN ROCK-PAPER-SCISSORS CLASSIFICATION - FINAL RESULTS")
print("=" * 65)

print(f"\nDevice: {DEVICE}")

print(f"\nStandard Test Accuracy:")
print(f"{test_accuracy:.2f}%")

print("\nCustom Image Predictions:")

for result in custom_results:

    print(
        f"  {result['filename']} -> "
        f"{result['predicted_class']} "
        f"({result['confidence']:.2f}%)"
    )

print("\nModel file:")
print(MODEL_PATH)

print("\nNumber of custom images:")
print(len(custom_results))

print("\nClasses:")
print(", ".join(CLASS_NAMES))

print("\nAssignment pipeline completed.")

In [ ]:
# ============================================================
# CELL 29: ASSIGNMENT REQUIREMENT CHECK
# ============================================================

print("=" * 65)
print("ASSIGNMENT REQUIREMENT CHECK")
print("=" * 65)

checks = {
    "GitHub repository cloned automatically":
        os.path.exists(REPO_DIR),

    "Custom dataset exists":
        os.path.exists(CUSTOM_DATA_DIR),

    "10 custom images":
        len(custom_images) == 10,

    "CNN uses nn.Module":
        isinstance(model, nn.Module),

    "Conv2d used":
        any(
            isinstance(layer, nn.Conv2d)
            for layer in model.modules()
        ),

    "ReLU used":
        any(
            isinstance(layer, nn.ReLU)
            for layer in model.modules()
        ),

    "MaxPool2d used":
        any(
            isinstance(layer, nn.MaxPool2d)
            for layer in model.modules()
        ),

    "Linear used":
        any(
            isinstance(layer, nn.Linear)
            for layer in model.modules()
        ),

    "Model .pth exists":
        os.path.exists(MODEL_PATH),

    "Standard test evaluation completed":
        len(all_test_predictions) > 0,

    "Confusion matrix generated":
        cm.shape == (3, 3),

    "Custom predictions completed":
        len(custom_results) == len(custom_images)
}

for requirement, status in checks.items():

    symbol = "PASS" if status else "FAIL"

    print(
        f"[{symbol}] {requirement}"
    )

print("\nDone.")